# E6 | Model Clustering K-Means
Segmentar incidentes em 4 clusters (A/B/C/D) para atuacao preventiva

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import os, pandas as pd, numpy as np
from sqlalchemy import create_engine
from dotenv import load_dotenv
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score
import mlflow

load_dotenv()
print('Setup OK')

In [ ]:
# RDS Connection
RDS_HOST = os.getenv('RDS_HOST')
RDS_USER = os.getenv('RDS_USER')
RDS_PASSWORD = os.getenv('RDS_PASSWORD')
RDS_DATABASE = os.getenv('RDS_DATABASE')

engine = create_engine(f'postgresql://{RDS_USER}:{RDS_PASSWORD}@{RDS_HOST}:5432/{RDS_DATABASE}')
print('RDS Connected')

In [ ]:
# Ler cluster dataset
df = pd.read_sql('SELECT * FROM gold.ml_cluster_dataset', engine)
print(f'Loaded {len(df)} records')
print(f'Columns: {df.shape[1]}')

In [ ]:
# Preparar features
feature_cols = [c for c in df.columns if c not in ['incident_id', 'cluster', 'data_abertura']]
X = df[feature_cols].fillna(0)

# Normalizar
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f'Features: {len(feature_cols)}')
print(f'Scaled shape: {X_scaled.shape}')

In [ ]:
# Encontrar k otimo (teste com k=2-8)
silhouette_scores = []
db_scores = []
ks = range(2, 9)

for k in ks:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    
    sil_score = silhouette_score(X_scaled, labels)
    db_score = davies_bouldin_score(X_scaled, labels)
    
    silhouette_scores.append(sil_score)
    db_scores.append(db_score)
    
    print(f'k={k}: Silhouette={sil_score:.4f}, DB={db_score:.4f}')

In [ ]:
# Usar k=4 (conforme requisito do desafio)
k = 4
model = KMeans(n_clusters=k, random_state=42, n_init=10)
print(f'Training K-Means with k={k}...')
labels = model.fit_predict(X_scaled)

sil_score = silhouette_score(X_scaled, labels)
db_score = davies_bouldin_score(X_scaled, labels)

print(f'Silhouette Score: {sil_score:.4f}')
print(f'Davies-Bouldin Index: {db_score:.4f}')

In [ ]:
# Adicionar clusters ao dataset
df['cluster_pred'] = labels
cluster_names = {0: 'A', 1: 'B', 2: 'C', 3: 'D'}
df['cluster_label'] = df['cluster_pred'].map(cluster_names)

print('Cluster distribution:')
print(df['cluster_label'].value_counts().sort_index())

In [ ]:
# Perfil dos clusters
cluster_profile = df.groupby('cluster_label')[feature_cols].mean()
print('Cluster Profile:')
print(cluster_profile.iloc[:, :5].round(2))

In [ ]:
# MLflow
mlflow.set_experiment('kmeans_clustering')
with mlflow.start_run():
    mlflow.log_params({
        'n_clusters': k,
        'n_features': len(feature_cols),
        'scaler': 'StandardScaler'
    })
    mlflow.log_metrics({
        'silhouette_score': float(sil_score),
        'davies_bouldin_index': float(db_score)
    })
    mlflow.sklearn.log_model(model, 'kmeans_model')
    print('Logged to MLflow')